# BNPL Default Prediction — Data Preparation (v2 — Rerun-Safe)
## Notebook 03: Temporal Split, Imputation, Encoding, Save

**What changed from v1:** every transformation is now a **pure function**
(input dataframe → output dataframe), not a repeated mutation of one
shared `train_df` variable across 12 cells. A single **PIPELINE** cell
near the end calls every function in sequence and rebuilds everything
from scratch.

**Why this matters:** in v1, re-running an earlier cell (like the split)
silently wiped out columns added by later cells, because they all kept
reassigning the same `train_df` name. That made the notebook's behavior
depend on *which order you clicked cells*, not just what's written on
the page — a classic, very common notebook bug.

**How to use this notebook safely:**
- Exploration cells (marked `# EXPLORE`) are safe to re-run anytime —
  they only read, never feed into the saved output.
- The single source of truth is the `run_pipeline()` function defined
  near the end. Re-run **that one cell** any time you want fresh,
  correct output — it rebuilds train/val/test from raw data every time,
  in the right order, with no dependency on prior cell execution history.
- If anything ever looks wrong: **Kernel → Restart → Run All** is always
  a safe reset, but with this structure you shouldn't need it as often.

**Locked feature set from Notebook 02 (17 features):**
`dti, fico_range_low, revol_util, annual_inc, loan_amnt, int_rate, sub_grade,
term, emp_length, home_ownership, verification_status, purpose, delinq_2yrs,
inq_last_6mths, open_acc, pub_rec, revol_bal`


## Section 1: Setup and Load Raw Data (run once per session)

In [20]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Libraries loaded")

Libraries loaded


In [21]:
FILE_PATH = '../data/raw/accepted_2007_to_2018Q4.csv.gz'

FEATURES = [
    'dti', 'fico_range_low', 'revol_util', 'annual_inc', 'loan_amnt',
    'int_rate', 'sub_grade', 'term', 'emp_length', 'home_ownership',
    'verification_status', 'purpose', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal'
]
LOAD_COLS = FEATURES + ['issue_d', 'loan_status']

NUMERIC_FEATURES = ['dti', 'fico_range_low', 'revol_util', 'annual_inc',
                     'loan_amnt', 'int_rate', 'delinq_2yrs', 'inq_last_6mths',
                     'open_acc', 'pub_rec', 'revol_bal']
CATEGORICAL_FEATURES = ['sub_grade', 'term', 'home_ownership',
                         'verification_status', 'purpose']
NOMINAL_COLS = ['home_ownership', 'verification_status', 'purpose']

EMP_LENGTH_MAP = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
SUB_GRADES_ORDERED = [f"{letter}{num}" for letter in "ABCDEFG" for num in range(1, 6)]
SUB_GRADE_MAP = {sg: i for i, sg in enumerate(SUB_GRADES_ORDERED)}

REVOL_UTIL_HARD_CAP = 100.0

# This raw load is expensive — keep it isolated to ONE cell, loaded ONCE.
# Re-running THIS cell is fine and intentional if you want a totally
# fresh start. Nothing downstream depends on it being run more than once
# per kernel session.
df_raw = pd.read_csv(FILE_PATH, usecols=LOAD_COLS, low_memory=False)
print(f"Raw rows loaded: {len(df_raw):,}")
print(f"Columns loaded: {df_raw.shape[1]}")

Raw rows loaded: 2,260,701
Columns loaded: 19


---
## Section 2: Pipeline Functions

Every function below takes a dataframe IN and returns a NEW dataframe
OUT. None of them mutate a shared variable across cells. This is what
makes the pipeline safe to re-run in any order — the only thing that
matters is calling them in the right sequence inside `run_pipeline()`,
which we do explicitly in Section 3.


In [22]:
def filter_and_target(df_in: pd.DataFrame) -> pd.DataFrame:
    """Keep only resolved loans, build binary target, parse time axis."""
    d = df_in[df_in['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
    d['default'] = (d['loan_status'] == 'Charged Off').astype(int)
    d = d.drop(columns=['loan_status'])
    d['issue_d'] = pd.to_datetime(d['issue_d'], format='%b-%Y', errors='coerce')
    d['issue_year'] = d['issue_d'].dt.year
    return d


def temporal_split(df_in: pd.DataFrame):
    """Split into train/val/test + two held-out windows for later phases."""
    train = df_in[df_in['issue_year'].isin([2013, 2014, 2015])].copy()
    val   = df_in[df_in['issue_year'] == 2016].copy()
    test  = df_in[df_in['issue_year'] == 2017].copy()
    prod_2018      = df_in[df_in['issue_year'] == 2018].copy()
    crisis_2008_09 = df_in[df_in['issue_year'].isin([2008, 2009])].copy()
    return train, val, test, prod_2018, crisis_2008_09


def fit_imputation_values(train: pd.DataFrame) -> dict:
    """Calculate median imputation values from TRAIN ONLY."""
    cols = NUMERIC_FEATURES + ['emp_length_num']
    # emp_length_num must exist before fitting — caller ensures order
    return {col: float(train[col].median()) for col in cols}


def fit_outlier_caps(train: pd.DataFrame) -> dict:
    """Calculate outlier caps from TRAIN ONLY. revol_util uses a hard
    business-logic cap (it's a percentage, mathematically bounded 0-100);
    everything else uses a 99th-percentile cap."""
    caps = {}
    for col in ['annual_inc', 'revol_bal', 'dti']:
        caps[col] = float(train[col].quantile(0.99))
    caps['revol_util'] = REVOL_UTIL_HARD_CAP
    return caps


def fit_nominal_categories(train: pd.DataFrame) -> dict:
    """Record the categories seen in TRAIN for each nominal column —
    this becomes the FIXED encoding schema for val/test/production."""
    return {col: sorted(train[col].dropna().unique().tolist())
            for col in NOMINAL_COLS}


def apply_emp_length(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d['emp_length_num'] = d['emp_length'].map(EMP_LENGTH_MAP)
    return d


def apply_imputation(df_in: pd.DataFrame, impute_values: dict) -> pd.DataFrame:
    d = df_in.copy()
    for col, val in impute_values.items():
        d[col + '_was_missing'] = d[col].isnull().astype(int)
        d[col] = d[col].fillna(val)
    return d


def apply_sub_grade_encoding(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d['sub_grade_encoded'] = d['sub_grade'].map(SUB_GRADE_MAP)
    return d


def apply_term_encoding(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d['term_num'] = d['term'].str.extract(r'(\d+)').astype(float)
    return d


def apply_one_hot(df_in: pd.DataFrame, train_categories: dict) -> pd.DataFrame:
    """One-hot encode using FIXED categories from train. Any category
    not seen in train maps to an all-zero row (no crash, no leakage)."""
    d = df_in.copy()
    parts = [d]
    for col, cats in train_categories.items():
        ohe = pd.DataFrame(index=d.index)
        for cat in cats:
            ohe[f"{col}_{cat}"] = (d[col] == cat).astype(int)
        parts.append(ohe)
    return pd.concat(parts, axis=1)


def apply_outlier_caps(df_in: pd.DataFrame, caps: dict) -> pd.DataFrame:
    d = df_in.copy()
    for col, cap_val in caps.items():
        d[col + '_capped'] = d[col].clip(upper=cap_val)
    return d


def build_final_columns(train_categories: dict, impute_values: dict) -> list:
    onehot_cols = [f"{col}_{cat}" for col, cats in train_categories.items()
                   for cat in cats]
    was_missing_cols = [c + '_was_missing' for c in impute_values.keys()]
    capped_replacements = ['annual_inc', 'revol_bal', 'revol_util', 'dti']

    base_numeric = [c for c in NUMERIC_FEATURES if c not in capped_replacements]
    capped_cols  = [c + '_capped' for c in capped_replacements]

    return (base_numeric + capped_cols +
            ['emp_length_num', 'sub_grade_encoded', 'term_num'] +
            onehot_cols + was_missing_cols)


def assemble_split(df_in: pd.DataFrame, impute_values: dict,
                   caps: dict, train_categories: dict,
                   final_cols: list) -> pd.DataFrame:
    """Apply the full transformation chain to one split (train/val/test)
    using parameters that were fit on TRAIN ONLY, then select final columns."""
    d = apply_emp_length(df_in)
    d = apply_imputation(d, impute_values)
    d = apply_sub_grade_encoding(d)
    d = apply_term_encoding(d)
    d = apply_one_hot(d, train_categories)
    d = apply_outlier_caps(d, caps)

    keep_cols = final_cols + ['default', 'issue_d', 'issue_year']
    return d[keep_cols].copy()

print("Pipeline functions defined.")

Pipeline functions defined.


---
## Section 3: Run The Pipeline (THE single source of truth)

**Re-run this one cell any time** to rebuild `train_final`, `val_final`,
`test_final` completely from scratch, in the correct order, using only
`df_raw` from Section 1. It does not matter what other cells you've run
before this, or in what order — as long as `df_raw` exists, this always
produces correct, consistent output.


In [23]:
def run_pipeline(df_raw: pd.DataFrame):
    # 1. Filter + target
    df_clean = filter_and_target(df_raw)

    # 2. Temporal split (raw, no transformations applied yet)
    train_raw, val_raw, test_raw, prod_2018_raw, crisis_raw = temporal_split(df_clean)

    # 3. Add emp_length_num BEFORE fitting imputation (median needs it to exist)
    train_with_emp = apply_emp_length(train_raw)

    # 4. Fit every transformation parameter on TRAIN ONLY
    impute_values    = fit_imputation_values(train_with_emp)
    outlier_caps      = fit_outlier_caps(train_raw)
    train_categories  = fit_nominal_categories(train_raw)
    final_cols        = build_final_columns(train_categories, impute_values)

    # 5. Apply the full chain to each split using train-fit parameters
    train_final = assemble_split(train_raw, impute_values, outlier_caps,
                                  train_categories, final_cols)
    val_final   = assemble_split(val_raw, impute_values, outlier_caps,
                                  train_categories, final_cols)
    test_final  = assemble_split(test_raw, impute_values, outlier_caps,
                                  train_categories, final_cols)

    fitted_params = {
        "impute_values": impute_values,
        "outlier_caps": outlier_caps,
        "train_categories": train_categories,
        "final_cols": final_cols,
    }

    held_out = {"prod_2018_raw": prod_2018_raw, "crisis_raw": crisis_raw}

    return train_final, val_final, test_final, fitted_params, held_out


# ── Run it ───────────────────────────────────────────────────────────────────
train_final, val_final, test_final, fitted_params, held_out = run_pipeline(df_raw)

print(f"Train final shape: {train_final.shape}")
print(f"Val final shape:   {val_final.shape}")
print(f"Test final shape:  {test_final.shape}")

Train final shape: (733451, 50)
Val final shape:   (293095, 50)
Test final shape:  (169300, 50)


---
## Section 4: Verification (EXPLORE — safe to re-run anytime)


In [24]:
# EXPLORE — class balance across splits
print("Class balance check across splits (should be roughly similar ~80/20):")
for name, d in [('Train', train_final), ('Val', val_final), ('Test', test_final)]:
    pct = d['default'].value_counts(normalize=True).mul(100).round(1)
    print(f"  {name}: {dict(pct)}")

Class balance check across splits (should be roughly similar ~80/20):
  Train: {0: np.float64(81.2), 1: np.float64(18.8)}
  Val: {0: np.float64(76.7), 1: np.float64(23.3)}
  Test: {0: np.float64(76.9), 1: np.float64(23.1)}


In [25]:
# EXPLORE — null check
print("Null check in final model columns:")
for name, d in [('train', train_final), ('val', val_final), ('test', test_final)]:
    nulls = d[fitted_params['final_cols']].isnull().sum().sum()
    print(f"  {name}: {nulls} nulls")

Null check in final model columns:
  train: 0 nulls
  val: 0 nulls
  test: 0 nulls


In [26]:
# EXPLORE — outlier caps applied
print("Outlier caps (fit on TRAIN only):")
for col, val in fitted_params['outlier_caps'].items():
    method = "hard business-logic cap" if col == 'revol_util' else "99th percentile cap"
    print(f"  {col:<15} cap = {val:>12,.2f}   ({method})")

print()
print("Max values AFTER capping (train) — sanity check:")
for col in fitted_params['outlier_caps']:
    print(f"  {col}_capped   max = {train_final[col + '_capped'].max():,.2f}")

Outlier caps (fit on TRAIN only):
  annual_inc      cap =   250,000.00   (99th percentile cap)
  revol_bal       cap =    93,283.00   (99th percentile cap)
  dti             cap =        37.57   (99th percentile cap)
  revol_util      cap =       100.00   (hard business-logic cap)

Max values AFTER capping (train) — sanity check:
  annual_inc_capped   max = 250,000.00
  revol_bal_capped   max = 93,283.00
  dti_capped   max = 37.57
  revol_util_capped   max = 100.00


In [27]:
# EXPLORE — final leakage re-check on processed features
final_numeric_check = (
    [c for c in NUMERIC_FEATURES if c not in ['annual_inc', 'revol_bal',
                                               'revol_util', 'dti']] +
    ['annual_inc_capped', 'revol_bal_capped', 'revol_util_capped',
     'dti_capped', 'emp_length_num', 'sub_grade_encoded', 'term_num']
)

print("Final correlation check — processed features vs default (train):")
print("-" * 55)
for col in final_numeric_check:
    if col in train_final.columns:
        corr = train_final[col].corr(train_final['default'])
        flag = "  << INVESTIGATE" if abs(corr) > 0.5 else ""
        print(f"  {col:<25} {corr:+.3f}{flag}")
print()
print("Expected: all values modest (roughly -0.3 to +0.3).")

Final correlation check — processed features vs default (train):
-------------------------------------------------------
  fico_range_low            -0.122
  loan_amnt                 +0.059
  int_rate                  +0.257
  delinq_2yrs               +0.014
  inq_last_6mths            +0.065
  open_acc                  +0.034
  pub_rec                   +0.020
  annual_inc_capped         -0.065
  revol_bal_capped          -0.025
  revol_util_capped         +0.052
  dti_capped                +0.110
  emp_length_num            -0.007
  sub_grade_encoded         +0.271
  term_num                  +0.207

Expected: all values modest (roughly -0.3 to +0.3).


In [28]:
# EXPLORE — class imbalance numbers for documentation
n_neg = (train_final['default'] == 0).sum()
n_pos = (train_final['default'] == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"Train class counts — Fully Paid: {n_neg:,} | Charged Off: {n_pos:,}")
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.3f}")
print()
print(f"Final feature count: {len(fitted_params['final_cols'])}")

Train class counts — Fully Paid: 595,463 | Charged Off: 137,988
scale_pos_weight for XGBoost: 4.315

Final feature count: 47


---
## Section 5: Save Outputs

Safe to re-run — always saves whatever is currently in
`train_final` / `val_final` / `test_final`, which only changes
when Section 3's pipeline cell is re-run.


In [29]:
os.makedirs('../data/processed', exist_ok=True)

train_final.to_parquet('../data/processed/train.parquet', index=False)
val_final.to_parquet('../data/processed/val.parquet', index=False)
test_final.to_parquet('../data/processed/test.parquet', index=False)

print("Saved:")
print("  ../data/processed/train.parquet")
print("  ../data/processed/val.parquet")
print("  ../data/processed/test.parquet")

Saved:
  ../data/processed/train.parquet
  ../data/processed/val.parquet
  ../data/processed/test.parquet


In [30]:
n_neg = (train_final['default'] == 0).sum()
n_pos = (train_final['default'] == 1).sum()
scale_pos_weight = float(n_neg / n_pos)

data_prep_config = {
    "feature_set_version": "v2_rerun_safe",
    "source_notebook_02_table": "../reports/feature_selection_table.csv",

    "temporal_split": {
        "train_years": [2013, 2014, 2015],
        "val_year": 2016,
        "test_year": 2017,
        "held_out_for_later": {
            "production_simulation": [2018],
            "crisis_simulation": [2008, 2009]
        }
    },

    "features": {
        "numeric": NUMERIC_FEATURES,
        "categorical_ordinal": ["sub_grade"],
        "categorical_nominal": NOMINAL_COLS,
        "binary": ["term"],
        "engineered": ["emp_length_num"]
    },

    "imputation": {
        "method": "median (calculated on TRAIN split only)",
        "values": fitted_params['impute_values'],
        "missing_flag_added": True
    },

    "encoding": {
        "sub_grade": {"method": "ordinal", "mapping": SUB_GRADE_MAP},
        "term": {"method": "numeric_extraction"},
        "nominal_categories_from_train": fitted_params['train_categories'],
        "unseen_category_handling": "all-zero one-hot row (no crash)",
        "rejected_alternatives": {
            "tfidf": "wrong data shape — single categorical labels, not free text",
            "embeddings": "only justified for high-cardinality features in neural nets; "
                          "low cardinality here (3-14 categories) makes one-hot the "
                          "standard, interpretable, defensible choice"
        }
    },

    "outlier_handling": {
        "method": "99th percentile cap (TRAIN only), except revol_util (hard cap at 100)",
        "caps": fitted_params['outlier_caps']
    },

    "class_imbalance": {
        "train_class_counts": {"fully_paid": int(n_neg), "charged_off": int(n_pos)},
        "strategy": "class_weighting",
        "scale_pos_weight": scale_pos_weight,
        "applied_at": "training time, not in this notebook"
    },

    "final_model_columns": fitted_params['final_cols'],
    "final_shapes": {
        "train": list(train_final.shape),
        "val": list(val_final.shape),
        "test": list(test_final.shape)
    }
}

os.makedirs('../reports', exist_ok=True)
with open('../reports/data_prep_config.json', 'w') as f:
    json.dump(data_prep_config, f, indent=2)

print("Saved: ../reports/data_prep_config.json")

Saved: ../reports/data_prep_config.json


---
## Summary

| Step | What Happened |
|------|---------------|
| Row filtering | Kept Fully Paid / Charged Off only, built binary `default` target |
| Temporal split | Train 2013-15 / Val 2016 / Test 2017 — 2018 and 2008-09 held out for later |
| Missing values | Median imputation (train-fit) + `was_missing` flags |
| `emp_length` | Converted text to 0-10 numeric scale |
| Categorical encoding | Ordinal for `sub_grade`, one-hot (train-fixed categories) for nominal features |
| Outliers | 99th percentile caps (train-fit); hard cap at 100 for `revol_util` |
| Class imbalance | Documented — class weighting via `scale_pos_weight`, applied at training time |
| Leakage re-check | Final correlation check on processed features — all within expected range |
| Saved outputs | `train/val/test.parquet` + `data_prep_config.json` |

**Rerun-safety note:** all of the above is produced by `run_pipeline()`
in Section 3 — one function, one cell, no dependency on what order you
ran other cells in. Re-running Section 3 always gives you correct,
reproducible output from `df_raw`.

**Next notebook:** `04_model_training.ipynb` — baseline (Logistic Regression)
vs XGBoost vs LightGBM comparison, MLflow experiment tracking, threshold
selection based on business cost (not F1-maximization).
